In [0]:
bronze_output = dbutils.jobs.taskValues.get(taskKey = "Bronze", key = "bronze_output")

# Access the data
catalog = bronze_output.get("catalog", "")
schema_1 = bronze_output.get("schema_1", "")
schema_2 = bronze_output.get("schema_2", "")

In [0]:
%run ./01-config


In [0]:
# Import core PySpark SQL functions (e.g., current_timestamp, from_json, window functions)
from pyspark.sql import functions as F
# Import Window for defining partitioned and ordered operations (e.g., ranking records)
from pyspark.sql.window import Window
from pyspark.sql.functions import from_unixtime

In [0]:
#   Construct absolute paths for raw data ingestion and streaming checkpoints
# - `landing_zone`: Base directory where raw source files (CSV/JSON) are staged for ingestion

base_dir_data = spark.sql("describe external location `data-zone`").select("url").collect()[0][0]
base_dir_checkpoint = spark.sql("describe external location `checkpoints-zone`").select("url").collect()[0][0]

In [0]:
# Switch to the catalog
spark.sql("USE CATALOG dev")

# Create the schema if it doesn't exist
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")



In [0]:
# - `checkpoint_base`: Directory to store streaming query offsets and state (required for fault tolerance)

TRAIN_CHECKPOINT = base_dir_checkpoint + "/silver/train_movements"
WEATHER_CHECKPOINT = base_dir_checkpoint + "/silver/weather"
CITY_ROUTE_CHECKPOINT = base_dir_checkpoint + "/silver/city_routes"
OPERATORS_CHECKPOINT = base_dir_checkpoint + "/silver/operator"
STATIONS_CHECKPOINT = base_dir_checkpoint + "/silver/station"
FACT_TRAIN = base_dir_checkpoint + "/silver/fact_train"
DIM_OPERATOR = base_dir_checkpoint + "/silver/dim_operator"
DIM_DATE = base_dir_checkpoint + "/silver/dim_date"

In [0]:
# === SILVER LAYER (Cleaned & Conformed Tables) ===
# These tables contain validated, deduplicated, and properly typed data.
# Timestamps are converted to `timestamp` type; keys are standardized.

# Creates a Silver `train_movements` table with cleaned train_movement data.
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{schema_2}.train_movements(
    train_id STRING,
    actual_timestamp TIMESTAMP,
    gbtt_timestamp TIMESTAMP,
    planned_timestamp TIMESTAMP,
    loc_stanox STRING,
    planned_event_type STRING,
    event_type STRING,
    event_source STRING,
    correction_ind BOOLEAN,
    offroute_ind BOOLEAN,
    direction_ind STRING,
    line_ind STRING,
    platform STRING,
    route STRING,
    train_service_code STRING,
    division_code STRING,
    toc_id STRING,
    timetable_variation INT,
    variation_status STRING,
    next_report_stanox STRING,
    next_report_run_time INT,
    train_terminated BOOLEAN,
    delay_monitoring_point BOOLEAN,
    reporting_stanox STRING,
    auto_expected BOOLEAN
)
USING DELTA
""");

In [0]:

# Creates a Silver `weather` table with cleaned weather data.
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{schema_2}.weather (
    city STRING,
    temperature DOUBLE,
    humidity INT,
    pressure INT,
    visibility INT,
    wind_speed DOUBLE,
    wind_direction INT,
    sunrise TIMESTAMP,
    sunset TIMESTAMP,
    weather_description STRING,
    cloudiness INT,
    event_time TIMESTAMP
)
USING DELTA
""");


In [0]:

# Creates a Silver `city_route` table with cleaned city_route data.
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{schema_2}.city_route (
    city STRING,
    route STRING
)
USING DELTA
""");

In [0]:

# Creates a Silver `train_operators` table with cleaned train_operator data.
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{schema_2}.train_operators (
    operating_company STRING,
    business_code STRING,
    toc_id STRING,
    atoc_code STRING
)
USING DELTA
""");

In [0]:
# Creates a Silver `station` table with cleaned station data.

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{schema_2}.station (
    stanox STRING,
    station STRING,
    sector_code STRING,
    route STRING
)
USING DELTA
""");

In [0]:
# Creates a Silver `fact_train_performance` table with cleaned fact_train_performance data.
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{schema_2}.fact_train_performance (
    date_key            INT,
    train_id            STRING,
    loc_stanox          STRING,
    next_report_stanox  STRING,
    reporting_stanox    STRING,
    toc_id              STRING,
    city                STRING,
    route               STRING,
    current_station     STRING,
    next_station        STRING,
    platform            STRING,
    direction           STRING,
    actual_timestamp    TIMESTAMP,
    planned_timestamp   TIMESTAMP,
    variation_status    STRING,
    temperature         DOUBLE,
    wind_speed          DOUBLE
)
USING DELTA         
""");

In [0]:
# Creates a Silver `dim_operators` table with cleaned dim_operators data.
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{schema_2}.dim_operators(
    operating_company   STRING,
    business_code       STRING,
    toc_id              STRING,
    atoc_code           STRING,
    effective_start_date  DATE,
    effective_end_date    DATE,
    is_current            BOOLEAN
)
USING DELTA          
""");

In [0]:

# Creates a Silver `dim_date` table with cleaned dim_date data.
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {catalog}.{schema_2}.dim_date(
        date_key INT,
        full_date DATE,
        day INT,
        day_of_week INT,
        day_name STRING,
        week_of_year INT,
        month INT,
        month_name STRING,
        quarter INT,
        year INT,
        is_weekend BOOLEAN,
        season STRING,
        fiscal_year INT,
        fiscal_quarter INT
    )
    USING DELTA
    """);

In [0]:
def upserter(df_micro_batch, batch_id, merge_query, temp_view):
    """
    Generic helper to execute a Delta MERGE operation on a micro-batch DataFrame.

    - Creates a temporary view from the micro-batch so it can be referenced in SQL MERGE
    - Executes the provided `merge_query`
    - Logs batch completion for observability

    Intended for reuse across Silver tables via foreachBatch.
    """
    df_micro_batch.createOrReplaceTempView(temp_view)
    df_micro_batch._jdf.sparkSession().sql(merge_query)
    print(f"Batch {batch_id} for {temp_view} processed.")


In [0]:
def upsert_train_silver(catalog: str, schema_1: str, schema_2: str, once=True, processing_time="15 seconds", startingVersion=0):
    """
    Creates/updates the `train` Silver table from Kafka train messages
    stored in `kafka_multiplex_bz`.

    - Parses JSON payload
    - Normalizes timestamps
    - Deduplicates events per train/location/time
    - MERGEs into Silver train table
    """

    merge_query = f"""
    MERGE INTO {catalog}.{schema_2}.train_movements a
    USING train_cdc b
    ON  a.train_id = b.train_id
    AND a.loc_stanox = b.loc_stanox
    AND a.actual_timestamp = b.actual_timestamp

    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
    """

    schema = """ train_id STRING, actual_timestamp STRING, loc_stanox STRING, gbtt_timestamp STRING, planned_timestamp STRING,
        planned_event_type STRING, event_type STRING, event_source STRING, correction_ind BOOLEAN, offroute_ind BOOLEAN, direction_ind STRING,
        line_ind STRING, platform STRING, route STRING, train_service_code STRING, division_code STRING, toc_id STRING,timetable_variation INT,
        variation_status STRING, next_report_stanox STRING, next_report_run_time INT, train_terminated BOOLEAN, delay_monitoring_point BOOLEAN,
        reporting_stanox STRING, auto_expected BOOLEAN
    """


    df_cdc = (
        spark.readStream
            .option("startingVersion", startingVersion)
            .option("ignoreDeletes", True)
            .table(f"{catalog}.{schema_1}.train_movements_raw")
            .filter("topic = 'train-movements'")
            .select(F.from_json(F.col("raw_payload"), schema).alias("v"))
            .select("v.*")
            .withColumn("actual_timestamp", F.to_timestamp((F.col("actual_timestamp") / 1000).cast("timestamp")))
            .withColumn("gbtt_timestamp", F.to_timestamp((F.col("gbtt_timestamp") / 1000).cast("timestamp")))
            .withColumn("planned_timestamp", F.to_timestamp((F.col("planned_timestamp") / 1000).cast("timestamp")))
            .filter(F.col("train_id").isNotNull() & F.col("actual_timestamp").isNotNull() & F.col("loc_stanox").isNotNull())
            .withWatermark("actual_timestamp", "2 minutes")
            .dropDuplicates(["train_id", "loc_stanox", "actual_timestamp"])
    )

    stream_writer = (
        df_cdc.writeStream
            .foreachBatch(lambda df, id: upserter(df,id,merge_query,temp_view="train_cdc"))
            .outputMode("append")
            .option("checkpointLocation", TRAIN_CHECKPOINT)
            .queryName("train_silver_stream")
    )

    if once:
        return stream_writer.trigger(availableNow=True).start()
    else:
        return stream_writer.trigger(processingTime=processing_time).start()





In [0]:
def upsert_city_route_silver(catalog: str, schema_1: str, schema_2: str, once=True, processing_time="15 seconds", startingVersion=0):
    """
    Creates/updates the `train` Silver table from Kafka train messages
    stored in `kafka_multiplex_bz`.

    - Parses JSON payload
    - Normalizes timestamps
    - Deduplicates events per train/location/time
    - MERGEs into Silver train table
    """

    merge_query = f"""
    MERGE INTO {catalog}.{schema_2}.city_route a
    USING city_route_cdc b
    ON  a.city = b.city
    AND a.route = b.route

    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
    """


    schema = """
    city STRING,
    route STRING
    """

    df_cdc = (
    spark.readStream
        .option("startingVersion", startingVersion)
        .option("ignoreDeletes", True)
        .table(f"{catalog}.{schema_1}.city_routes_raw")
        .select(
            F.col("city"),
            F.col("route")
        )
        .filter(
            F.col("city").isNotNull() &
            F.col("route").isNotNull()
        )
        .dropDuplicates(["city", "route"])
    )

    stream_writer = (
    df_cdc.writeStream
        .foreachBatch(lambda df, id: upserter(df, id, merge_query, temp_view="city_route_cdc"))
        .outputMode("append")
        .option("checkpointLocation", CITY_ROUTE_CHECKPOINT)
        .queryName("city_route_silver_stream")
    )

    if once:
        return stream_writer.trigger(availableNow=True).start()
    else:
        return stream_writer.trigger(processingTime=processing_time).start()

In [0]:
def upsert_operators_silver(catalog: str, schema_1: str, schema_2: str, once=True, processing_time="15 seconds", startingVersion=0):
    """
    Creates/updates the `train` Silver table from Kafka train messages
    stored in `kafka_multiplex_bz`.

    - Parses JSON payload
    - Normalizes timestamps
    - Deduplicates events per train/location/time
    - MERGEs into Silver train table
    """

    merge_query = f"""
    MERGE INTO dev.silver.train_operators AS a
    USING operators_cdc AS b
    ON  a.business_code = b.business_code

    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
    """


    schema = """
    Company_Name STRING,
    Business_Code STRING,
    Sector_Code STRING,
    ATOC_Code STRING
    """

    df_cdc = (
    spark.readStream
        .option("startingVersion", 0)
        .option("ignoreDeletes", True)
        .table(f"{catalog}.{schema_1}.station_raw")

        # Select required columns
        .select(
            F.col("Company_Name").alias("operating_company"),
            F.col("Business_Code").alias("business_code"),
            F.col("Sector_Code").alias("toc_id"),
            F.col("Atoc_Code").alias("atoc_code")
        )

        # Data quality filter
        .filter(
            F.col("business_code").isNotNull()
        )

        # Deduplicate inside batch
        .dropDuplicates(["business_code"])
    )
    

    stream_writer = (
    df_cdc.writeStream
        .foreachBatch(lambda df, id: upserter(df, id, merge_query, temp_view="operators_cdc"))
        .outputMode("append")
        .option("checkpointLocation", OPERATORS_CHECKPOINT)
        .queryName("operators_silver_stream")
    )

    if once:
        return stream_writer.trigger(availableNow=True).start()
    else:
        return stream_writer.trigger(processingTime=processing_time).start()

In [0]:
def upsert_station_silver(catalog: str, schema_1: str, schema_2: str, once=True, processing_time="15 seconds", startingVersion=0):
    """
    Creates/updates the `train` Silver table from Kafka train messages
    stored in `kafka_multiplex_bz`.

    - Parses JSON payload
    - Normalizes timestamps
    - Deduplicates events per train/location/time
    - MERGEs into Silver train table
    """

    merge_query = f"""
    MERGE INTO {catalog}.{schema_2}.station a
    USING station_cdc b
    ON  a.stanox = b.stanox

    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
    """


    schema = """
    Company_Name STRING,
    Business_Code STRING,
    Sector_Code STRING,
    ATOC_Code STRING
    """

    df_cdc = (
    spark.readStream
        .option("startingVersion", 0)
        .option("ignoreDeletes", True)
        .table(f"{catalog}.{schema_1}.stations_code_raw")

        # Select required columns
        .select(
            F.col("Company_Name").alias("stanox"),
            F.col("Business_Code").alias("station"),
            F.col("Sector_Code").alias("sector_code"),
            F.col("Atoc_Code").alias("route")
        )

        # Data quality filter
        .filter(
            F.col("stanox").isNotNull()
        )

        # Deduplicate inside batch
        .dropDuplicates(["stanox"])
    )

    stream_writer = (
    df_cdc.writeStream
        .foreachBatch(lambda df, id: upserter(df, id, merge_query, temp_view="station_cdc"))
        .outputMode("append")
        .option("checkpointLocation", STATIONS_CHECKPOINT)
        .queryName("station_silver_stream")
    )

    if once:
        return stream_writer.trigger(availableNow=True).start()
    else:
        return stream_writer.trigger(processingTime=processing_time).start()

In [0]:
def upsert_weather_silver(catalog: str, schema_1: str, schema_2: str, once=True, processing_time="15 seconds", startingVersion=0):
    """
    Creates/updates the `train` Silver table from Kafka train messages
    stored in `kafka_multiplex_bz`.

    - Parses JSON payload
    - Normalizes timestamps
    - Deduplicates events per train/location/time
    - MERGEs into Silver weather table
    """

    merge_query = f"""
    MERGE INTO {catalog}.{schema_2}.weather a
    USING weather_cdc b
    ON a.city = b.city
    AND a.event_time = b.event_time
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
    """

    raw_schema = """
    coord STRING,
    weather ARRAY<STRUCT<id: INT, main: STRING, description: STRING, icon: STRING>>,
    base STRING,
    main STRUCT<temp: DOUBLE, feels_like: DOUBLE, temp_min: DOUBLE, temp_max: DOUBLE, pressure: INT, humidity: INT, sea_level: INT, grnd_level: INT>,
    visibility INT,
    wind STRUCT<speed: DOUBLE, deg: INT>,
    clouds STRUCT<all: INT>,
    dt LONG,
    sys STRUCT<type: INT, id: INT, country: STRING, sunrise: LONG, sunset: LONG>,
    timezone INT,
    id LONG,
    name STRING,
    cod INT
    """


    df_weather_cdc = (
    spark.readStream
        .option("startingVersion", startingVersion)
        .option("ignoreDeletes", True)
        .table(f"{catalog}.{schema_1}.weather_raw")
        .filter("topic = 'weather-updates'")
        .select(F.from_json(F.col("raw_payload"), raw_schema).alias("v"))
        .select(
            F.col("v.name").alias("city"),
            F.col("v.main.temp").alias("temperature"),
            F.col("v.main.humidity").alias("humidity"),
            F.col("v.main.pressure").alias("pressure"),
            F.col("v.visibility").alias("visibility"),
            F.col("v.wind.speed").alias("wind_speed"),
            F.col("v.wind.deg").alias("wind_direction"),
            F.from_unixtime(F.col("v.sys.sunrise")).alias("sunrise"),
            F.from_unixtime(F.col("v.sys.sunset")).alias("sunset"),
            F.expr("v.weather[0].description").alias("weather_description"),
            F.col("v.clouds.all").alias("cloudiness"),
            F.from_unixtime(F.col("v.dt")).alias("event_time")
        )
        .withColumn("sunrise", F.col("sunrise").cast("timestamp"))
        .withColumn("sunset", F.col("sunset").cast("timestamp"))
        .withColumn("event_time", F.col("event_time").cast("timestamp"))
        .filter(
            F.col("city").isNotNull() &
            F.col("event_time").isNotNull()
        )
        .withWatermark("event_time", "2 minutes")
        .dropDuplicates(["city", "event_time"])
    )


    stream_writer = (
        df_weather_cdc.writeStream
            .foreachBatch(lambda df, id: upserter(df,id,merge_query,temp_view="weather_cdc"))
            .outputMode("update")
            .option("checkpointLocation", WEATHER_CHECKPOINT)
            .queryName("weather_silver_stream")
    )

    if once:
        return stream_writer.trigger(availableNow=True).start()
    else:
        return stream_writer.trigger(processingTime=processing_time).start()

In [0]:
from pyspark.sql.functions import (col, expr, coalesce, lit, lower, trim)

def upsert_fact_train_stream(once=True, processing_time="15 seconds", startingVersion=0):

    merge_query = f"""
    MERGE INTO {catalog}.{schema_2}.fact_train_performance a
    USING fact_train_delta b
    ON  a.train_id = b.train_id
    AND a.actual_timestamp = b.actual_timestamp
    AND a.loc_stanox = b.loc_stanox
    WHEN NOT MATCHED THEN INSERT *
    """

    # ===============================
    # 1️⃣ Static Dimensions
    # ===============================

    station = spark.read.table(
        f"{catalog}.{schema_2}.station"
    ).alias("station")

    sc_next = spark.read.table(
        f"{catalog}.{schema_2}.station"
    ).alias("sc_next")

    city_route = spark.read.table(
        f"{catalog}.{schema_2}.city_route"
    ).alias("city_route")

    # ===============================
    # 2️⃣ Streaming Train Movements
    # ===============================

    movements = (
    spark.readStream
        .table(f"{catalog}.{schema_2}.train_movements")
        .alias("movements")
        .withWatermark("actual_timestamp", "30 minutes")
        # Add the date_key column in YYYYMMDD integer format
        .withColumn("date_key", F.date_format("actual_timestamp", "yyyyMMdd").cast("int"))
    )

    # ===============================
    # 3️⃣ Streaming Weather (10-min feed)
    # ===============================

    weather = (
        spark.readStream
        .table(f"{catalog}.{schema_2}.weather")
        .alias("weather")
        .withWatermark("event_time", "30 minutes")
    )

    # ===============================
    # 4️⃣ Join Static Dimensions First
    # ===============================

    df = movements.join(
        station,
        col("movements.loc_stanox") == col("station.stanox"),
        "left"
    )

    df = df.join(
        sc_next,
        col("movements.next_report_stanox") == col("sc_next.stanox"),
        "left"
    )

    df = df.join(
        city_route,
        col("station.route") == col("city_route.route"),
        "left"
    )

    # ===============================
    # 5️⃣ Stream–Stream Join (±15 min window)
    # ===============================

    df = df.join(
        weather,
        (
            col("city_route.city") == col("weather.city")
        ) &
        (
            col("weather.event_time").between(
                col("movements.actual_timestamp") - expr("INTERVAL 1 HOUR"),
                col("movements.actual_timestamp") + expr("INTERVAL 1 HOUR")
            )
        ),
        "leftOuter"
    )

    # ===============================
    # 6️⃣ Final Select
    # ===============================

    df_delta = df.select(
        col("date_key"),
        col("movements.train_id").alias("train_id"),
        col("movements.loc_stanox").alias("loc_stanox"),
        col("movements.next_report_stanox").alias("next_report_stanox"),
        col("movements.reporting_stanox").alias("reporting_stanox"),
        col("movements.toc_id").alias("toc_id"),
        col("city_route.city").alias("city"),
        col("movements.actual_timestamp").alias("actual_timestamp"),
        col("movements.planned_timestamp").alias("planned_timestamp"),
        col("movements.variation_status").alias("variation_status"),
        col("movements.platform").alias("platform"),
        col("movements.direction_ind").alias("direction"),
        coalesce(col("station.station"), lit("Ending Station")).alias("current_station"),
        coalesce(col("sc_next.station"), lit("Unknown")).alias("next_station"),
        col("station.route").alias("route"),
        col("weather.temperature").alias("temperature"),
        col("weather.wind_speed").alias("wind_speed")
    )

    # ===============================
    # 7️⃣ Write with MERGE
    # ===============================

    def process_batch(batch_df, batch_id):
        upserter(batch_df, batch_id, merge_query, "fact_train_delta")

    stream_writer = (
        df_delta.writeStream
        .foreachBatch(process_batch)
        .outputMode("append")
        .option("checkpointLocation", FACT_TRAIN)
        .queryName("fact_train_weather_10min_stream")
    )

    if once:
        return stream_writer.trigger(availableNow=True).start()
    else:
        return stream_writer.trigger(processingTime=processing_time).start()


In [0]:
def upsert_operator_scd2(once=True, processing_time="30 seconds"):
    """
    Incrementally upserts operators into `dim_operator` using SCD2 logic.
    - Marks old rows as inactive when attributes change
    - Inserts new row with effective_start_date = load_time
    """

    merge_query = f"""
    MERGE INTO {catalog}.{schema_2}.dim_operators AS target
    USING operator_cdc AS source
    ON target.business_code = source.business_code
       AND target.is_current = TRUE

    WHEN MATCHED AND (
        target.operating_company != source.operating_company OR
        target.toc_id != source.toc_id OR
        target.atoc_code != source.atoc_code
    )
    THEN UPDATE SET
        target.is_current = FALSE,
        target.effective_end_date = source.load_time

    WHEN NOT MATCHED
    THEN INSERT (
        business_code,
        operating_company,
        toc_id,
        atoc_code,
        effective_start_date,
        effective_end_date,
        is_current
    )
    VALUES (
        source.business_code,
        source.operating_company,
        source.toc_id,
        source.atoc_code,
        source.load_time,
        NULL,
        TRUE
    )
    """

    # Read streaming source and prepare CDC batch
    df_cdc = (
        spark.readStream
            .table(f"{catalog}.{schema_2}.train_operators")
            .withColumn("load_time", F.current_timestamp())
            .dropDuplicates(["business_code"])
    )

    # foreachBatch upsert using existing upserter function
    stream_writer = (
        df_cdc.writeStream
            .foreachBatch(lambda df, batch_id: upserter(
                df.withColumn("effective_start_date", F.col("load_time"))
                  .withColumn("effective_end_date", F.lit(None).cast("timestamp"))
                  .withColumn("is_current", F.lit(True)),
                batch_id,
                merge_query,
                temp_view="operator_cdc"
            ))
            .outputMode("append")
            .option("checkpointLocation", DIM_OPERATOR)
            .queryName("dim_operator_scd2_stream")
    )


    if once:
        return stream_writer.trigger(availableNow=True).start()
    else:
        return stream_writer.trigger(processingTime=processing_time).start()




In [0]:
def upsert_dim_date(once=True, processing_time="30 seconds", startingVersion=0):
    """
    Builds and upserts a Dim Date table from train_movements timestamps.
    
    - Uses actual_timestamp and planned_timestamp from train_movements
    - Generates date attributes: year, month, day, day_of_week
    - Upserts into dim_date table using foreachBatch and upserter
    """

    # ===============================
    # 1️⃣ Read streaming source
    # ===============================
    df_train = (
        spark.readStream
            .table(f"{catalog}.{schema_2}.train_movements")
            .select(
                F.coalesce(
                    F.to_date("actual_timestamp"),
                    F.to_date("planned_timestamp")
                ).alias("full_date")
            )
            .filter(F.col("full_date").isNotNull())
            .dropDuplicates(["full_date"])
    )

    # ===============================
    # 2️⃣ Prepare dim_date columns
    # ===============================
    def prepare_dim_date(batch_df, batch_id):
        # Generate all necessary columns for dim_date
        df_dim_date = (
        batch_df
        .withColumn("date_key", F.date_format("full_date", "yyyyMMdd").cast("int"))
        .withColumn("day", F.dayofmonth("full_date"))
        .withColumn("day_of_week", F.dayofweek("full_date"))
        .withColumn("day_name", F.date_format("full_date", "EEEE"))
        .withColumn("week_of_year", F.weekofyear("full_date"))
        .withColumn("month", F.month("full_date"))
        .withColumn("month_name", F.date_format("full_date", "MMMM"))
        .withColumn("quarter", F.quarter("full_date"))
        .withColumn("year", F.year("full_date"))
        .withColumn("is_weekend",
                    F.when(F.dayofweek("full_date").isin([1,7]), True)
                     .otherwise(False))
        .withColumn("season",
                    F.when(F.month("full_date").isin([12,1,2]), "Winter")
                     .when(F.month("full_date").isin([3,4,5]), "Spring")
                     .when(F.month("full_date").isin([6,7,8]), "Summer")
                     .otherwise("Autumn"))
        .withColumn("fiscal_year",
                    F.when(F.month("full_date") >= 4, F.year("full_date"))
                     .otherwise(F.year("full_date") - 1))
        .withColumn("fiscal_quarter",
                    F.when(F.month("full_date").between(4,6), 1)
                     .when(F.month("full_date").between(7,9), 2)
                     .when(F.month("full_date").between(10,12), 3)
                     .otherwise(4))
    )

        # Upsert into dim_date table using existing upserter
        merge_query = f"""
        MERGE INTO {catalog}.{schema_2}.dim_date AS target
        USING tmp_dim_date AS source
        ON target.full_date = source.full_date
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
        """
        upserter(df_dim_date, batch_id, merge_query, temp_view="tmp_dim_date")

    # ===============================
    # 3️⃣ Start the streaming write
    # ===============================
    stream_writer = (
        df_train.writeStream
            .foreachBatch(prepare_dim_date)
            .outputMode("append")
            .option("checkpointLocation", DIM_DATE)
            .queryName("dim_date_stream")
    )

    if once:
        return stream_writer.trigger(availableNow=True).start()
    else:
        return stream_writer.trigger(processingTime=processing_time).start()


In [0]:
upsert_train_silver( catalog=catalog, schema_1=schema_1, schema_2=schema_2, once=True, processing_time="15 seconds", startingVersion=0)

In [0]:
upsert_operators_silver(catalog=catalog, schema_1=schema_1, schema_2=schema_2, once=True, processing_time="15 seconds", startingVersion=0)

In [0]:
upsert_operators_silver(catalog=catalog, schema_1=schema_1, schema_2=schema_2, once=True, processing_time="15 seconds", startingVersion=0)

In [0]:
upsert_station_silver(catalog=catalog, schema_1=schema_1, schema_2=schema_2, once=True, processing_time="15 seconds", startingVersion=0)

In [0]:
upsert_weather_silver( catalog=catalog, schema_1=schema_1, schema_2=schema_2, once=True, processing_time="15 seconds", startingVersion=0)

In [0]:
# %sql
# SELECT *
# FROM dev.silver.fact_train_performance
# WHERE temperature is not null
# LIMIT 5;

In [0]:
upsert_city_route_silver( catalog=catalog, schema_1=schema_1, schema_2=schema_2, once=True, processing_time="15 seconds", startingVersion=0)

In [0]:
upsert_fact_train_stream(once=True, processing_time="15 seconds", startingVersion=0)

In [0]:
upsert_operator_scd2(once=True, processing_time="30 seconds")

In [0]:
upsert_dim_date(once=True, processing_time="30 seconds", startingVersion=0)

In [0]:
# %sql
# SELECT *
# FROM dev.silver.fact_train_performance
# WHERE temperature is not null
# LIMIT 5;

In [0]:
def assert_zero(catalog, schema_2, table_name, condition):
    """
    Assert that no records match a bad condition.
    """
    print(f"Checking {table_name} for condition: {condition}")

    count = (
        spark.read.table(f"{catalog}.{schema_2}.{table_name}")
        .where(condition)
        .count()
    )

    assert count == 0, f"{count} invalid records found in {table_name} where {condition}"

    print("✔ Passed")


def assert_not_empty(catalog, schema_2, table_name):
    """
    Ensure table is not empty.
    """
    print(f"Checking {table_name} is not empty")

    count = spark.read.table(f"{catalog}.{schema_2}.{table_name}").count()

    assert count > 0, f"{table_name} is empty"

    print(f"✔ {count} records found")

In [0]:

print("\n🔎 Validating Silver Layer")

table = "fact_train_movements"

assert_not_empty(catalog, schema_2, "city_route")
assert_not_empty(catalog, schema_2, "station")
assert_not_empty(catalog, schema_2, "train_movements")
assert_not_empty(catalog, schema_2, "weather")
assert_not_empty(catalog, schema_2, "train_operators")
assert_not_empty(catalog, schema_2, "dim_operators")
assert_not_empty(catalog, schema_2, "dim_date")
assert_not_empty(catalog, schema_2, "fact_train_performance")


# # Timestamp must exist
# assert_zero(catalog, schema_2, "train_movements", "actual_timestamp IS NULL")

# assert_zero(catalog, schema_2, "train_movements", "planned_timestamp IS NULL")

# # Station information required
# assert_zero(catalog, schema_2, table, "current_station IS NULL")
# assert_zero(catalog, schema_2, table, "next_station IS NULL")

print("✔ Silver validation completed")

In [0]:
# Convert string → real datetime.date

output_data = {
    "catalog": "dev",
    "schema_1": "bronze",
    "schema_2": "silver",
    "schema_3": "gold"
}

dbutils.jobs.taskValues.set(key = "silver_output", value = output_data)